In [26]:
import numpy as np
import pandas as pd
import torch
import random

def set_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(721)


In [27]:
import os
import torch
import torch.nn as nn
import random
from pathlib import Path
from torch.utils.data import Dataset
import torch.nn.functional as F
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, random_split

In [28]:
class MIBCI2aDataset(Dataset):
    
    def get_features(self, filePath):
        # implement the getFeatures method
        """
        read all the preprocessed data from the file path, read it using np.load,
        and concatenate them into a single numpy array
        """
        self.filePath = filePath
        npy_files = [os.path.join(self.filePath, file)
                    for file in os.listdir(self.filePath)
                    if file.endswith(".npy")]
        arrays = [np.load(file) for file in npy_files]
        features = np.concatenate(arrays, axis = 0)
        #concat with axis 0 
        self.L = features.shape[0]
        print(f"features with shape:{features.shape}")
        return features

    def get_labels(self, filePath):
        # implement the getLabels method
        """
        read all the preprocessed labels from the file path, read it using np.load,
        and concatenate them into a single numpy array
        """
        self.filePath = filePath
        npy_files = [os.path.join(self.filePath, file)
                    for file in os.listdir(self.filePath)
                    if file.endswith('.npy')]
        arrays = [np.load(file) for file in npy_files]
        labels = np.concatenate(arrays, axis = 0)
        #concat with axis 0 
        print(f"labels with shape:{labels.shape}")
        return labels

    def __init__(self, mode, _exp_name):
        super(MIBCI2aDataset).__init__()
        # remember to change the file path according to different experiments
        assert mode in ['train', 'test', 'finetune']
        self.mode = mode
        if mode == 'train':
            # subject dependent: ./dataset/SD_train/features/ and ./dataset/SD_train/labels/
            # leave-one-subject-out: ./dataset/LOSO_train/features/ and ./dataset/LOSO_train/labels/
            #self.features = self.get_features(filePath='/kaggle/input/lab2-data/lab2/dataset/LOSO_train/features')
            #self.labels = self.get_labels(filePath='/kaggle/input/lab2-data/lab2/dataset/LOSO_train/labels')
            
            self.features = self.get_features(filePath='/kaggle/input/lab2-data/Lab 2 - EEG Motor Imagery Classification/lab2/dataset/SD_train/features')
            self.labels = self.get_labels(filePath='/kaggle/input/lab2-data/Lab 2 - EEG Motor Imagery Classification/lab2/dataset/SD_train/labels')
        if mode == 'finetune':
            # finetune: ./dataset/FT/features/ and ./dataset/FT/labels/
            self.features = self.get_features(filePath='/kaggle/input/lab2-data/Lab 2 - EEG Motor Imagery Classification/lab2/dataset/FT/features')
            self.labels = self.get_labels(filePath='/kaggle/input/lab2-data/Lab 2 - EEG Motor Imagery Classification/lab2/dataset/FT/labels')
        if mode == 'test':
            if _exp_name == 'FT':
                _exp_name = 'LOSO'
            # subject dependent: ./dataset/SD_test/features/ and ./dataset/SD_test/labels/
            # leave-one-subject-out and finetune: ./dataset/LOSO_test/features/ and ./dataset/LOSO_test/labels/
            self.features = self.get_features(filePath=f'/kaggle/input/lab2-data/Lab 2 - EEG Motor Imagery Classification/lab2/dataset/{_exp_name}_test/features')
            self.labels = self.get_labels(filePath=f'/kaggle/input/lab2-data/Lab 2 - EEG Motor Imagery Classification/lab2/dataset/{_exp_name}_test/labels')

    def __len__(self):
        # implement the len method
        return self.L

    def __getitem__(self, idx):
        # implement the getitem method
        feature, label = torch.from_numpy(self.features[idx]).double(), int(self.labels[idx])
        return feature, label

In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SquareLayer(nn.Module):
    def __init__(self):
        super(SquareLayer, self).__init__()

    def forward(self, x):
        return x ** 2

class SCCNet(nn.Module):
    def __init__(self, C = 22, Nt = 1, Nu = 22, dropout = 0.5):
        super().__init__()
        self.CNN1 = nn.Conv2d(1, Nu, (C, Nt), stride = 1, padding = 0)
        #torch.nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, 
                        #dilation=1, groups=1, bias=True, padding_mode='zeros', device=None, dtype=None)
        self.batch_norm1 = nn.BatchNorm2d(Nu)
        
        #self.CNN2 = nn.Conv2d(1, 20, (Nu, 12), stride = 1, padding=(0, 6))
        self.CNN2 = nn.Conv2d(Nu, 20, kernel_size=(1, 12), padding=(0, 6))
        self.batch_norm2 = nn.BatchNorm2d(20)
        
        self.avg_pooling = nn.AvgPool2d((1, 62), stride=(1, 12))
        
        self.square = SquareLayer()
                                    
        self.fc = nn.Linear(640, 4)
        
        self.dropout = nn.Dropout(dropout)
        
        #self.softmax = nn.Softmax(dim=1)
        
    def forward(self, x):
        x = x.unsqueeze(1)
        # x.shape = B x 1 x 22 x 438
        x = self.CNN1(x)
        #x.shape = B x Nu x 1 x 438
        x = self.batch_norm1(x)
        #x = self.square(x)
        #x = self.dropout(x)
        #x.shape = B x Nu x 438
        #x.shape = B x 1 x Nu x 438
        x = self.CNN2(x)
        #x.shape = B, 20, 32
        x = self.batch_norm2(x)
        x = x.squeeze(2)
        x = self.square(x)
        #x.shape = B x 20 x 439
        x = self.dropout(x)
        #print(x.shape)
        x = self.avg_pooling(x)
        #print(x.shape)
        #x.shape = B x 20 x 438
        x = torch.log(x)
        x = x.view(-1,640)
        #x.shape = B x 140
        x = self.fc(x)
        
        #x = self.softmax(x)
        #x.shape = B x 4
        return x

In [30]:
def tester(_exp_name, batch_size, n_workers):
    test_set = MIBCI2aDataset('test', _exp_name)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    test_loader = DataLoader(
        test_set,
        batch_size = batch_size,
        num_workers = n_workers,
        drop_last = False,
        pin_memory = True,
    )
    #model_best = SCCNet().double().to(device)
    model_best = SCCNet().double().to(device)
    model_best.load_state_dict(torch.load(f"/kaggle/input/lab2-data/{_exp_name}_best.ckpt"))
    test_loss = []
    test_accs = []
    model_best.eval()
    criterion = nn.CrossEntropyLoss()
    
    
    for batch in tqdm(test_loader):
        features, labels = batch
        
        with torch.no_grad():
            logits = model_best(features.to(device))
        
        loss = criterion(logits, labels.to(device))
        acc = (logits.argmax(dim=-1) == labels.to(device)).float().mean()
        
        test_loss.append(loss.item())
        test_accs.append(acc)
        
    test_loss = sum(test_loss) / len(test_loss)
    test_acc = sum(test_accs) / len(test_accs)
    
    print(f"  Test |  loss = {test_loss:.5f}, acc = {test_acc:.5f}")

In [31]:
batch_size = 150

In [32]:
tester("SD", batch_size, 0)
print(f"batch_size = {batch_size}")

features with shape:(2304, 22, 438)
labels with shape:(2304,)


  0%|          | 0/16 [00:00<?, ?it/s]

  Test |  loss = 1.02581, acc = 0.60560
batch_size = 150


In [33]:
tester("LOSO", batch_size, 0)
print(f"batch_size = {batch_size}")

features with shape:(288, 22, 438)
labels with shape:(288,)


  0%|          | 0/2 [00:00<?, ?it/s]

  Test |  loss = 1.65814, acc = 0.52942
batch_size = 150


In [34]:
tester("FT", batch_size, 0)
print(f"batch_size = {batch_size}")

features with shape:(288, 22, 438)
labels with shape:(288,)


  0%|          | 0/2 [00:00<?, ?it/s]

  Test |  loss = 0.70961, acc = 0.72638
batch_size = 150
